In [ ]:
!pip install wbgapi yfinance pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 66.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import wbgapi as wb
import yfinance as yf
import pycountry
import warnings
warnings.filterwarnings('ignore')

# --- 1. CARGAR Y LIMPIAR EL DATASET ---
print("1. Cargando y limpiando datos...")
df = pd.read_csv("Mineria_Integrada_Consistente.csv", sep=";")

df['Pais_Destino_Limpio'] = df['Pais_Destino'].str.strip().str.upper()
df['Pais_Destino_Limpio'] = df['Pais_Destino_Limpio'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
remplazos = {'REP. DOMINICANA': 'REPUBLICA DOMINICANA', 'MYANMAR (EX BIRMANIA)': 'MYANMAR', 'TAIWAN (FORMOSA)': 'TAIWAN'}
df['Pais_Destino_Limpio'] = df['Pais_Destino_Limpio'].replace(remplazos)

# --- 2. CREAR REGIONES ---
regiones = {
    'Norteamérica': ['ESTADOS UNIDOS', 'CANADA', 'MEXICO'],
    'Mercosur': ['BRASIL', 'ARGENTINA', 'PARAGUAY', 'URUGUAY', 'VENEZUELA', 'BOLIVIA'],
    'Unión Europea': ['BELGICA', 'HOLANDA', 'ITALIA', 'FRANCIA', 'ESPANA', 'ALEMANIA', 'AUSTRIA', 'DINAMARCA', 'GRECIA', 'FINLANDIA', 'PORTUGAL', 'POLONIA', 'LETONIA', 'LITUANIA', 'SUECIA', 'IRLANDA', 'CHIPRE', 'RUMANIA', 'LUXEMBURGO'],
    'Europa No-UE': ['REINO UNIDO', 'NORUEGA', 'SUIZA', 'RUSIA'],
    'Asia-Pacífico': ['CHINA', 'INDIA', 'VIETNAM', 'COREA DEL SUR', 'JAPON', 'AUSTRALIA', 'NUEVA ZELANDIA', 'MALASIA', 'TAIWAN', 'SINGAPUR', 'PAKISTAN', 'INDONESIA', 'SRI LANKA', 'FILIPINAS', 'MYANMAR', 'CAMBODIA', 'THAILANDIA', 'BANGLADESH', 'HONG KONG', 'COREA DEL NORTE', 'FIJI'],
    'Resto de LATAM y Caribe': ['COLOMBIA', 'PERU', 'COSTA RICA', 'ECUADOR', 'GUATEMALA', 'EL SALVADOR', 'PANAMA', 'REPUBLICA DOMINICANA', 'NICARAGUA', 'HONDURAS', 'HAITI', 'JAMAICA', 'GUYANA', 'TRINIDAD Y TOBAGO', 'PUERTO RICO', 'CUBA', 'BARBADOS', 'SURINAM', 'SANTA LUCIA', 'DOMINICA', 'ISLAS CAYMAN', 'BELICE'],
    'Medio Oriente y Norte de África (MENA)': ['ARABIA SAUDITA', 'EGIPTO', 'EMIRATOS ARABES UNIDOS', 'TURQUIA', 'ARGELIA', 'SUDAN', 'TUNEZ', 'MARRUECOS', 'LIBANO', 'KUWAIT', 'ISRAEL', 'SIRIA', 'JORDANIA', 'OMAN', 'IRAN', 'LIBIA'],
    'África Subsahariana': ['MAURICIO', 'NIGERIA', 'ETIOPIA', 'SENEGAL', 'BENIN', 'SUDAFRICA', 'KENIA', 'GAMBIA', 'MOZAMBIQUE', 'TOGO', 'SIERRA LEONA', 'ANGOLA', 'TANZANIA', 'GHANA', 'COSTA DE MARFIL']
}
mapeo_paises = {pais: region for region, paises in regiones.items() for pais in paises}
df['Region_Destino'] = df['Pais_Destino_Limpio'].map(mapeo_paises)

# --- 3. MAPEO ISO-3 Y DESCARGA BANCO MUNDIAL ---
print("2. Descargando datos del Banco Mundial...")
# Mapeo básico para los países principales (puedes extenderlo si la consola te avisa de nulos)
mapa_iso3 = {
    'CHINA': 'CHN', 'INDIA': 'IND', 'ESTADOS UNIDOS': 'USA', 'BELGICA': 'BEL',
    'HOLANDA': 'NLD', 'BRASIL': 'BRA', 'NORUEGA': 'NOR', 'ITALIA': 'ITA',
    'MEXICO': 'MEX', 'VIETNAM': 'VNM', 'COREA DEL SUR': 'KOR', 'JAPON': 'JPN',
    'AUSTRALIA': 'AUS', 'REINO UNIDO': 'GBR', 'FRANCIA': 'FRA', 'ARGENTINA': 'ARG',
    'ESPANA': 'ESP', 'NUEVA ZELANDIA': 'NZL', 'ARABIA SAUDITA': 'SAU', 'COLOMBIA': 'COL',
    'PERU': 'PER', 'COSTA RICA': 'CRI', 'BOLIVIA': 'BOL', 'URUGUAY': 'URY',
    'ECUADOR': 'ECU', 'GUATEMALA': 'GTM', 'PARAGUAY': 'PRY', 'MALASIA': 'MYS',
    'EL SALVADOR': 'SLV', 'ALEMANIA': 'DEU', 'TAIWAN': 'TWN', 'SINGAPUR': 'SGP',
    'PANAMA': 'PAN', 'AUSTRIA': 'AUT', 'CANADA': 'CAN', 'EGIPTO': 'EGY',
    'EMIRATOS ARABES UNIDOS': 'ARE', 'PAKISTAN': 'PAK', 'REPUBLICA DOMINICANA': 'DOM',
    'RUSIA': 'RUS', 'VENEZUELA': 'VEN', 'MAURICIO': 'MUS', 'INDONESIA': 'IDN',
    'NICARAGUA': 'NIC', 'SRI LANKA': 'LKA', 'NIGERIA': 'NGA', 'ETIOPIA': 'ETH',
    'SENEGAL': 'SEN', 'BENIN': 'BEN', 'SUDAFRICA': 'ZAF', 'KENIA': 'KEN',
    'DINAMARCA': 'DNK', 'HONDURAS': 'HND', 'GAMBIA': 'GMB', 'MOZAMBIQUE': 'MOZ',
    'FILIPINAS': 'PHL', 'HAITI': 'HTI', 'MYANMAR': 'MMR', 'CAMBODIA': 'KHM',
    'THAILANDIA': 'THA', 'JAMAICA': 'JAM', 'TURQUIA': 'TUR', 'ARGELIA': 'DZA',
    'GRECIA': 'GRC', 'FINLANDIA': 'FIN', 'SUDAN': 'SDN', 'GUYANA': 'GUY',
    'TUNEZ': 'TUN', 'TRINIDAD Y TOBAGO': 'TTO', 'TOGO': 'TGO', 'PUERTO RICO': 'PRI',
    'SIERRA LEONA': 'SLE', 'ANGOLA': 'AGO', 'TANZANIA': 'TZA', 'PORTUGAL': 'PRT',
    'CUBA': 'CUB', 'BARBADOS': 'BRB', 'MARRUECOS': 'MAR', 'POLONIA': 'POL',
    'GHANA': 'GHA', 'LIBANO': 'LBN', 'KUWAIT': 'KWT', 'LETONIA': 'LVA',
    'LITUANIA': 'LTU', 'SUIZA': 'CHE', 'SURINAM': 'SUR', 'ISRAEL': 'ISR',
    'SIRIA': 'SYR', 'COREA DEL NORTE': 'PRK', 'HONG KONG': 'HKG', 'CHIPRE': 'CYP',
    'SUECIA': 'SWE', 'IRLANDA': 'IRL', 'SANTA LUCIA': 'LCA', 'FIJI': 'FJI',
    'BANGLADESH': 'BGD', 'JORDANIA': 'JOR', 'DOMINICA': 'DMA', 'ISLAS CAYMAN': 'CYM',
    'OMAN': 'OMN', 'IRAN': 'IRN', 'RUMANIA': 'ROU', 'LIBIA': 'LBY', 'BELICE': 'BLZ',
    'COSTA DE MARFIL': 'CIV', 'LUXEMBURGO': 'LUX'
}
df['ISO3'] = df['Pais_Destino_Limpio'].map(mapa_iso3)
paises_iso3 = df['ISO3'].dropna().unique().tolist()
anios = df['Año'].unique().tolist()

indicadores_wb = {
    'NY.GDP.MKTP.CD': 'PIB_Pais_Destino',
    'NY.GDP.MKTP.KD.ZG': 'Crecimiento_PIB_Destino_%',
    'FP.CPI.TOTL.ZG': 'Inflacion_Destino_%'
}

wb_data = wb.data.DataFrame(indicadores_wb.keys(), paises_iso3, anios, numericTimeKeys=True).reset_index()
wb_melt = wb_data.melt(id_vars=['economy', 'series'], var_name='Año', value_name='Valor')
wb_pivot = wb_melt.pivot_table(index=['economy', 'Año'], columns='series', values='Valor').reset_index()
wb_pivot = wb_pivot.rename(columns={'economy': 'ISO3'}).rename(columns=indicadores_wb)

# --- 4. TIPO DE CAMBIO DESDE YAHOO FINANCE ---
print("3. Descargando Tipo de Cambio (USD/CLP)...")
usd_clp = yf.download('USDCLP=X', start=f"{min(anios)}-01-01", end=f"{max(anios)+1}-01-01", progress=False)
usd_clp['Año'] = usd_clp.index.year
tc_anual = usd_clp.groupby('Año')['Close'].mean().reset_index()
tc_anual.columns = ['Año', 'Tipo_Cambio_Promedio_USD_CLP']

# --- 5. INTEGRACIÓN FINAL ---
print("4. Integrando y guardando archivo final...")
df_integrado = pd.merge(df, wb_pivot, on=['ISO3', 'Año'], how='left')
df_integrado = pd.merge(df_integrado, tc_anual, on='Año', how='left')

# Limpiar columnas auxiliares
df_integrado = df_integrado.drop(columns=['Pais_Destino_Limpio', 'ISO3'])

# Guardar el nuevo archivo
df_integrado.to_csv("Mineria_Enriquecida_Completa.csv", sep=";", index=False,encoding='utf-8-sig')
print("¡ÉXITO! Se ha creado el archivo 'Mineria_Enriquecida_Completa.csv'")
df_integrado.head(3)

1. Cargando y limpiando datos...
2. Descargando datos del Banco Mundial...
3. Descargando Tipo de Cambio (USD/CLP)...
4. Integrando y guardando archivo final...
¡ÉXITO! Se ha creado el archivo 'Mineria_Enriquecida_Completa.csv'


,Producto,Pais_Destino,Año,Valor_FOB_Miles_USD,Volumen_Fisico_Ton,Produccion_Nacional_Ton,Precio_Unitario_USD_x_Ton,Region_Destino,Inflacion_Destino_%,PIB_Pais_Destino,Crecimiento_PIB_Destino_%,Tipo_Cambio_Promedio_USD_CLP
0,YODO (28012000),CHINA,2005,14734.39057,859.4,15346,17.144974,Asia-Pacífico,1.776414,2.317551e+12,11.453513,558.19081
1,YODO (28012000),INDIA,2005,17238.39025,953.1,15346,18.086654,Asia-Pacífico,4.246344,8.203838e+11,7.923431,558.19081
2,YODO (28012000),ESTADOS UNIDOS,2005,82801.87500,4919.7,15346,16.830676,Norteamérica,3.392747,1.303920e+13,3.483550,558.19081


In [ ]:
# --- 6. CÁLCULO DE CORRELACIONES REALES Y EXPORTACIÓN CORREGIDA ---
print("5. Calculando correlaciones reales (Sin alucinaciones de IA)...")

# Calculamos las correlaciones exactas (Ignorando nulos)
corr_pib = df_integrado['PIB_Pais_Destino'].corr(df_integrado['Volumen_Fisico_Ton'])
corr_tc = df_integrado['Tipo_Cambio_Promedio_USD_CLP'].corr(df_integrado['Valor_FOB_Miles_USD'])
corr_inf = df_integrado['Inflacion_Destino_%'].corr(df_integrado['Volumen_Fisico_Ton'])

print(f"Correlación PIB vs Volumen: {corr_pib:.3f}")
print(f"Correlación Tipo Cambio vs FOB: {corr_tc:.3f}")
print(f"Correlación Inflación vs Volumen: {corr_inf:.3f}")

print("\n6. Exportando archivo a prueba de fallos de Excel...")
# El parámetro decimal=',' es la magia que evita que Excel destruya los datos
df_integrado.to_csv("Mineria_Enriquecida_Corregida.csv", sep=";", decimal=",", index=False, encoding='utf-8-sig')

print("¡ÉXITO! Usa el archivo 'Mineria_Enriquecida_Corregida.csv' para el Streamlit.")

5. Calculando correlaciones reales (Sin alucinaciones de IA)...
Correlación PIB vs Volumen: 0.160
Correlación Tipo Cambio vs FOB: 0.033
Correlación Inflación vs Volumen: -0.008

6. Exportando archivo a prueba de fallos de Excel...
¡ÉXITO! Usa el archivo 'Mineria_Enriquecida_Corregida.csv' para el Streamlit.


In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset original
print("Cargando datos...")
df = pd.read_csv("Mineria_Enriquecida_Corregida.csv", sep=";", decimal=",", encoding="utf-8")

# 2. Arreglar formatos básicos
df['Año'] = df['Año'].astype(int)

# 3. Crear la agrupación por Familia de Producto
print("Agrupando minerales...")
df['Producto_Upper'] = df['Producto'].str.upper()

condiciones = [
    df['Producto_Upper'].str.contains('LITIO'),
    df['Producto_Upper'].str.contains('YOD'),
    df['Producto_Upper'].str.contains('POTAS|SALITRE')
]
opciones = ['Litio', 'Yodo', 'Potasio y Fertilizantes']

df['Familia_Producto'] = np.select(condiciones, opciones, default='Otros No Metálicos')

# Borrar la columna auxiliar
df.drop(columns=['Producto_Upper'], inplace=True)

# 4. Reordenar las columnas para que Familia_Producto quede al principio
columnas = df.columns.tolist()
# Movemos 'Familia_Producto' al inicio, justo después de 'Producto'
columnas.insert(1, columnas.pop(columnas.index('Familia_Producto')))
df = df[columnas]

# 5. Guardar el nuevo dataset corregido
# Usaremos sep=";" pero guardaremos los decimales nativos de Python (con punto)
# para que sea un CSV estándar internacional.
nombre_nuevo = "Mineria_Final.csv"
df.to_csv(nombre_nuevo, sep=";", index=False, encoding="utf-8")

print(f"¡Listo! Dataset corregido guardado exitosamente como: {nombre_nuevo}")

Cargando datos...
Agrupando minerales...
¡Listo! Dataset corregido guardado exitosamente como: Mineria_Final.csv
